# Notebook 01 — Crowd Density Estimation

**Goal:** Train and evaluate two architectures on ShanghaiTech (A+B) and JHU-Crowd++:
1. **CSRNet** (CVPR 2018 baseline)
2. **AdaptiveCSRNet** (ours: CBAM attention + multi-scale ASPP backend)

**Evaluation metrics:** MAE, MSE, PSNR, SSIM, GAME(0–3)

> All cells run automatically. Training state is checkpointed — interrupted runs resume automatically.

In [1]:
# Auto-detect repo root for JupyterLab / GCP / local execution

from pathlib import Path

import sys



def find_repo_root(start=None):

    start_path = Path(start or Path.cwd()).resolve()

    for candidate in [start_path, *start_path.parents]:

        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():

            return candidate

    return start_path



REPO_ROOT = find_repo_root()

DATA_ROOT = REPO_ROOT / 'data'

CKPT_ROOT = REPO_ROOT / 'checkpoints'

EXPS_ROOT = REPO_ROOT / 'experiments'

EXPS_ROOT.mkdir(exist_ok=True)

CKPT_ROOT.mkdir(exist_ok=True)



sys.path.insert(0, str(REPO_ROOT))



import torch

import torch.optim as optim

import numpy as np

import matplotlib.pyplot as plt

from tqdm.notebook import tqdm



from src.data_loaders import get_shanghaitech_loaders, get_jhu_loaders

from src.models import CSRNet, AdaptiveCSRNet

from src.losses import CombinedDensityLoss

from src.training import DensityTrainer

from src.evaluation import evaluate_density

from src.utils import show_density_map, plot_training_curves, make_results_table



DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

if torch.cuda.is_available():

    print(f'GPU: {torch.cuda.get_device_name(0)}')

    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

/opt/pytorch/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Device: cuda
GPU: NVIDIA A10G
VRAM: 23.7 GB


In [2]:
# ── Cell 2: Training configuration ───────────────────────────────────────────
CFG = {
    # Data
    'sha_part'    : 'A',           # 'A' or 'B'
    'target_size' : (576, 768),    # (H, W) — set smaller (e.g. 288, 384) to save memory
    'batch_size'  : 4,             # reduce if OOM
    'workers'     : 2,

    # Training
    'epochs_baseline'  : 100,
    'epochs_adaptive'  : 150,
    'lr'               : 1e-5,
    'lr_adaptive'      : 5e-5,
    'weight_decay'     : 5e-4,
    'patience'         : 20,
}

# Reduce sizes for quick prototyping with CPU
if DEVICE == 'cpu':
    CFG['target_size'] = (288, 384)
    CFG['epochs_baseline'] = 3
    CFG['epochs_adaptive'] = 3
    print('CPU mode: reduced to 3 epochs for demo run')

print('Configuration:', CFG)

Configuration: {'sha_part': 'A', 'target_size': (576, 768), 'batch_size': 4, 'workers': 2, 'epochs_baseline': 100, 'epochs_adaptive': 150, 'lr': 1e-05, 'lr_adaptive': 5e-05, 'weight_decay': 0.0005, 'patience': 20}


In [3]:
# ── Cell 3: Load ShanghaiTech data ────────────────────────────────────────────
print(f'Loading ShanghaiTech Part {CFG["sha_part"]}...')
train_loader, test_loader = get_shanghaitech_loaders(
    data_root=str(DATA_ROOT),
    part=CFG['sha_part'],
    batch_size=CFG['batch_size'],
    target_size=CFG['target_size'],
    num_workers=CFG['workers'],
)

print(f'Train batches: {len(train_loader)}  ({len(train_loader.dataset)} images)')
print(f'Test  batches: {len(test_loader)}  ({len(test_loader.dataset)} images)')

# Peek at one batch
imgs, density, counts = next(iter(train_loader))
print(f'Batch shapes  → imgs: {tuple(imgs.shape)}, density: {tuple(density.shape)}, counts: {counts[:4].numpy()}')

Loading ShanghaiTech Part A...
Train batches: 75  (300 images)
Test  batches: 182  (182 images)
Batch shapes  → imgs: (4, 3, 576, 768), density: (4, 1, 576, 768), counts: [1157.  393.  195.  284.]


In [4]:
# ── Cell 4: Train CSRNet baseline ─────────────────────────────────────────────
print('=' * 60)
print('TRAINING CSRNet BASELINE')
print('=' * 60)

model_csrnet = CSRNet(load_weights=True).to(DEVICE)
print(f'CSRNet parameters: {sum(p.numel() for p in model_csrnet.parameters()):,}')

optimizer_csrnet = optim.Adam(
    model_csrnet.parameters(),
    lr=CFG['lr'], weight_decay=CFG['weight_decay']
)
scheduler_csrnet = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_csrnet, factor=0.5, patience=10, min_lr=1e-7
)

trainer_csrnet = DensityTrainer(
    model=model_csrnet,
    optimizer=optimizer_csrnet,
    scheduler=scheduler_csrnet,
    device=DEVICE,
    experiment_name=f'csrnet_sha{CFG["sha_part"]}',
    save_dir=str(CKPT_ROOT),
    log_dir=str(REPO_ROOT / 'runs'),
)

# Resume if checkpoint exists
trainer_csrnet.load_checkpoint('last.pt')

history_csrnet = trainer_csrnet.train(
    train_loader, test_loader,
    epochs=CFG['epochs_baseline'],
    patience=CFG['patience'],
    metric_key='mae',
)

TRAINING CSRNet BASELINE


CSRNet parameters: 16,263,489
Loaded checkpoint from epoch 58, best=82.9798

Epoch 60/159
  [TRAIN] loss=0.0071  mae=112.7731
  [VAL  ] mae=86.3647  mse=134.3393  psnr=29.1743  ssim=0.7754  game0=86.3647  game1=24.1837  game2=6.9054  game3=2.0170

Epoch 61/159
  [TRAIN] loss=0.0071  mae=111.4285
  [VAL  ] mae=81.8839  mse=137.9341  psnr=29.2462  ssim=0.7892  game0=81.8839  game1=22.6891  game2=6.5171  game3=1.9243
  ✓ New best mae: 81.8839

Epoch 62/159
  [TRAIN] loss=0.0071  mae=106.6624
  [VAL  ] mae=86.0767  mse=150.3927  psnr=29.2687  ssim=0.7919  game0=86.0767  game1=23.5689  game2=6.6549  game3=1.9422

Epoch 63/159
  [TRAIN] loss=0.0072  mae=113.5380
  [VAL  ] mae=86.7200  mse=137.3892  psnr=29.1727  ssim=0.7725  game0=86.7200  game1=24.4234  game2=6.9852  game3=2.0396

Epoch 64/159
  [TRAIN] loss=0.0071  mae=108.7363
  [VAL  ] mae=82.4595  mse=141.2896  psnr=29.2587  ssim=0.7876  game0=82.4595  game1=22.9079  game2=6.5524  game3=1.9314

Epoch 65/159
  [TRAIN] loss=0.0071  mae=10

In [4]:
# ── Cell 4b: Load trained CSRNet (skip retraining) ───────────────────────────
print('Loading trained CSRNet from checkpoint...')

model_csrnet = CSRNet(load_weights=False).to(DEVICE)
print(f'CSRNet parameters: {sum(p.numel() for p in model_csrnet.parameters()):,}')

optimizer_csrnet = optim.Adam(
    model_csrnet.parameters(),
    lr=CFG['lr'], weight_decay=CFG['weight_decay']
)
scheduler_csrnet = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_csrnet, factor=0.5, patience=10, min_lr=1e-7
)

trainer_csrnet = DensityTrainer(
    model=model_csrnet,
    optimizer=optimizer_csrnet,
    scheduler=scheduler_csrnet,
    device=DEVICE,
    experiment_name=f'csrnet_sha{CFG["sha_part"]}',
    save_dir=str(CKPT_ROOT),
    log_dir=str(REPO_ROOT / 'runs'),
)

# Load the best checkpoint
trainer_csrnet.load_checkpoint('best.pt')
print(f'✓ Loaded best CSRNet checkpoint')

# Placeholder history so downstream cells work
history_csrnet = trainer_csrnet.history


Loading trained CSRNet from checkpoint...


CSRNet parameters: 16,263,489
Loaded checkpoint from epoch 70, best=81.7294
✓ Loaded best CSRNet checkpoint


In [5]:
# ── Cell 5: Evaluate CSRNet ───────────────────────────────────────────────────
trainer_csrnet.load_checkpoint('best.pt')
csrnet_results = evaluate_density(model_csrnet, test_loader, DEVICE)

print('\nCSRNet Results on SHA-' + CFG['sha_part'] + ':')
print('-' * 40)
for k, v in csrnet_results.items():
    print(f'  {k:8s}: {v:.4f}')

Loaded checkpoint from epoch 70, best=81.7294



CSRNet Results on SHA-A:
----------------------------------------
  mae     : 81.7294
  mse     : 140.3313
  psnr    : 29.2548
  ssim    : 0.7876
  game0   : 81.7294
  game1   : 22.8164
  game2   : 6.5829
  game3   : 1.9466


In [11]:
# ── Cell 6: Train AdaptiveCSRNet (our model) ──────────────────────────────────
print('=' * 60)
print('TRAINING AdaptiveCSRNet (OURS)')
print('=' * 60)

model_adaptive = AdaptiveCSRNet(load_weights=True, return_features=False).to(DEVICE)
n_params = sum(p.numel() for p in model_adaptive.parameters())
print(f'AdaptiveCSRNet parameters: {n_params:,}')

# Separate LR for pre-trained backbone vs new layers
backbone_params = list(model_adaptive.encoder.parameters())
new_params      = (list(model_adaptive.attention.parameters()) +
                   list(model_adaptive.backend.parameters()) +
                   list(model_adaptive.head.parameters()) +
                   list(model_adaptive.perspective_head.parameters()))

optimizer_adaptive = optim.AdamW([
    {'params': backbone_params, 'lr': CFG['lr_adaptive'] * 0.1},
    {'params': new_params,      'lr': CFG['lr_adaptive']},
], weight_decay=CFG['weight_decay'])

scheduler_adaptive = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_adaptive, T_max=CFG['epochs_adaptive'], eta_min=1e-7
)

loss_fn = CombinedDensityLoss(mse_weight=1.0, ssim_weight=0.5, tv_weight=1e-4)

trainer_adaptive = DensityTrainer(
    model=model_adaptive,
    optimizer=optimizer_adaptive,
    scheduler=scheduler_adaptive,
    loss_fn=loss_fn,
    device=DEVICE,
    experiment_name=f'adaptive_csrnet_sha{CFG["sha_part"]}',
    save_dir=str(CKPT_ROOT),
    log_dir=str(REPO_ROOT / 'runs'),
)

trainer_adaptive.load_checkpoint('last.pt')

history_adaptive = trainer_adaptive.train(
    train_loader, test_loader,
    epochs=CFG['epochs_adaptive'],
    patience=CFG['patience'],
    metric_key='mae',
)

TRAINING AdaptiveCSRNet (OURS)


AdaptiveCSRNet parameters: 9,481,092
Loaded checkpoint from epoch 60, best=74.5836

Epoch 62/211
  [TRAIN] loss=0.0036  mae=55.4312
  [VAL  ] mae=81.3586  mse=140.7889  psnr=29.1201  ssim=0.7989  game0=81.3586  game1=22.1302  game2=6.2387  game3=1.8369

Epoch 63/211
  [TRAIN] loss=0.0036  mae=53.0558
  [VAL  ] mae=75.4088  mse=126.7898  psnr=29.0876  ssim=0.7932  game0=75.4088  game1=20.9356  game2=6.0379  game3=1.8235

Epoch 64/211
  [TRAIN] loss=0.0035  mae=48.9355
  [VAL  ] mae=80.0330  mse=148.7903  psnr=29.1520  ssim=0.7963  game0=80.0330  game1=21.7347  game2=6.1492  game3=1.8386

Epoch 65/211
  [TRAIN] loss=0.0035  mae=57.0912
  [VAL  ] mae=79.5695  mse=131.3764  psnr=29.0191  ssim=0.7913  game0=79.5695  game1=21.9894  game2=6.2902  game3=1.8833

Epoch 66/211
  [TRAIN] loss=0.0036  mae=56.2923
  [VAL  ] mae=80.0191  mse=147.1490  psnr=29.1634  ssim=0.7922  game0=80.0191  game1=22.3741  game2=6.3779  game3=1.8906

Epoch 67/211
  [TRAIN] loss=0.0036  mae=63.3739
  [VAL  ] mae=82.0

In [12]:
# ── Cell 9b: Load trained AdaptiveCSRNet (skip retraining) ────────────────────
print('=' * 60)
print('LOADING TRAINED AdaptiveCSRNet')
print('=' * 60)

# load_weights=False because we are loading from our checkpoint anyway
model_adaptive = AdaptiveCSRNet(load_weights=False, return_features=False).to(DEVICE)
n_params = sum(p.numel() for p in model_adaptive.parameters())
print(f'AdaptiveCSRNet parameters: {n_params:,}')

# Separate LR for pre-trained backbone vs new layers
backbone_params = list(model_adaptive.encoder.parameters())
new_params      = (list(model_adaptive.attention.parameters()) +
                   list(model_adaptive.backend.parameters()) +
                   list(model_adaptive.head.parameters()) +
                   list(model_adaptive.perspective_head.parameters()))

optimizer_adaptive = optim.AdamW([
    {'params': backbone_params, 'lr': CFG['lr_adaptive'] * 0.1},
    {'params': new_params,      'lr': CFG['lr_adaptive']},
], weight_decay=CFG['weight_decay'])

scheduler_adaptive = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_adaptive, T_max=CFG['epochs_adaptive'], eta_min=1e-7
)

loss_fn = CombinedDensityLoss(mse_weight=1.0, ssim_weight=0.5, tv_weight=1e-4)

trainer_adaptive = DensityTrainer(
    model=model_adaptive,
    optimizer=optimizer_adaptive,
    scheduler=scheduler_adaptive,
    loss_fn=loss_fn,
    device=DEVICE,
    experiment_name=f'adaptive_csrnet_sha{CFG["sha_part"]}',
    save_dir=str(CKPT_ROOT),
    log_dir=str(REPO_ROOT / 'runs'),
)

# Load the best checkpoint instead of last.pt
trainer_adaptive.load_checkpoint('best.pt')
print('✓ Loaded best AdaptiveCSRNet checkpoint')

# Placeholder history so downstream cells still


LOADING TRAINED AdaptiveCSRNet


AdaptiveCSRNet parameters: 9,481,092
Loaded checkpoint from epoch 76, best=73.9194
✓ Loaded best AdaptiveCSRNet checkpoint


In [13]:
# ── Cell 7: Evaluate AdaptiveCSRNet ──────────────────────────────────────────
trainer_adaptive.load_checkpoint('best.pt')
adaptive_results = evaluate_density(model_adaptive, test_loader, DEVICE)

print('\nAdaptiveCSRNet Results on SHA-' + CFG['sha_part'] + ':')
print('-' * 40)
for k, v in adaptive_results.items():
    print(f'  {k:8s}: {v:.4f}')

Loaded checkpoint from epoch 76, best=73.9194



AdaptiveCSRNet Results on SHA-A:
----------------------------------------
  mae     : 73.9194
  mse     : 128.4377
  psnr    : 29.1506
  ssim    : 0.7953
  game0   : 73.9194
  game1   : 20.7864
  game2   : 5.9910
  game3   : 1.8116


In [14]:
# ── Cell 8: Comparison table ──────────────────────────────────────────────────
all_results = {
    'CSRNet (baseline)' : csrnet_results,
    'AdaptiveCSRNet (ours)': adaptive_results,
}

table = make_results_table(all_results, columns=['mae', 'mse', 'psnr', 'ssim', 'game0', 'game1', 'game2', 'game3'])
print(f'\n=== ShanghaiTech Part {CFG["sha_part"]} Results ===')
print(table)

# Save to file
result_file = EXPS_ROOT / f'density_sha{CFG["sha_part"]}_results.md'
result_file.write_text(f'# SHA-{CFG["sha_part"]} Density Estimation Results\n\n' + table)
print(f'\nSaved to {result_file}')


=== ShanghaiTech Part A Results ===
| Method | mae | mse | psnr | ssim | game0 | game1 | game2 | game3 |
|--------|--------|--------|--------|--------|--------|--------|--------|--------|
| CSRNet (baseline) | 81.73 | 140.33 | 29.25 | 0.79 | 81.73 | 22.82 | 6.58 | 1.95 |
| AdaptiveCSRNet (ours) | 73.92 | 128.44 | 29.15 | 0.80 | 73.92 | 20.79 | 5.99 | 1.81 |

Saved to /home/ubuntu/crowdvision/experiments/density_shaA_results.md


In [15]:
# ── Cell 9: Training curve comparison ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Training Curves — SHA-{CFG["sha_part"]}', fontsize=13)

for label, history, color in [
    ('CSRNet',          history_csrnet,  'blue'),
    ('AdaptiveCSRNet',  history_adaptive,'red'),
]:
    train_mae = [e.get('mae', e.get('loss', 0)) for e in history['train']]
    val_mae   = [e.get('mae', 0) for e in history['val']]
    axes[0].plot(train_mae, color=color, linestyle='--', label=f'{label} (train)', alpha=0.7)
    axes[1].plot(val_mae,   color=color, label=f'{label} (val)')

for ax, title in zip(axes, ['Train Loss', 'Val MAE']):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(EXPS_ROOT / f'density_sha{CFG["sha_part"]}_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [16]:
# ── Cell 10: Visual comparison on test samples ────────────────────────────────
import torchvision.transforms as T

model_csrnet.eval()
model_adaptive.eval()

# Get 3 test samples
test_iter = iter(test_loader)
n_show = min(3, len(test_loader))

denorm = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

for i in range(n_show):
    imgs, density_gt, counts_gt = next(test_iter)
    with torch.no_grad():
        pred_base = model_csrnet(imgs.to(DEVICE))
        pred_ours = model_adaptive(imgs.to(DEVICE))

    img_vis = denorm(imgs[0]).clamp(0, 1)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f'Test Sample {i+1}  |  GT Count: {counts_gt[0].item():.0f}', fontsize=12)

    axes[0].imshow(img_vis.permute(1,2,0).numpy())
    axes[0].set_title('Input'); axes[0].axis('off')

    axes[1].imshow(density_gt[0,0].numpy(), cmap='jet')
    axes[1].set_title(f'GT Density (sum={density_gt[0,0].sum():.0f})')
    axes[1].axis('off')

    p_base = pred_base[0,0].cpu().numpy()
    axes[2].imshow(p_base, cmap='jet')
    axes[2].set_title(f'CSRNet (count={p_base.sum():.0f})')
    axes[2].axis('off')

    p_ours = pred_ours[0,0].cpu().numpy()
    axes[3].imshow(p_ours, cmap='jet')
    axes[3].set_title(f'Ours (count={p_ours.sum():.0f})')
    axes[3].axis('off')

    plt.tight_layout()
    plt.savefig(EXPS_ROOT / f'density_sample_{i+1}.png', dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()

In [17]:
# ── Cell 11: Also train on JHU-Crowd++ (if available) ─────────────────────────
jhu_train_dir = DATA_ROOT / 'jhu_crowd_v2.0' / 'train'

if jhu_train_dir.exists():
    print('JHU-Crowd++ dataset found. Training AdaptiveCSRNet on JHU...')
    
    jhu_train, jhu_val, jhu_test = get_jhu_loaders(
        data_root=str(DATA_ROOT),
        batch_size=CFG['batch_size'],
        target_size=CFG['target_size'],
        num_workers=CFG['workers'],
    )
    print(f'JHU Train: {len(jhu_train.dataset)} | Val: {len(jhu_val.dataset)} | Test: {len(jhu_test.dataset)}')

    model_jhu = AdaptiveCSRNet(load_weights=True).to(DEVICE)
    opt_jhu = optim.AdamW(model_jhu.parameters(), lr=CFG['lr_adaptive'], weight_decay=CFG['weight_decay'])
    sched_jhu = optim.lr_scheduler.CosineAnnealingLR(opt_jhu, T_max=CFG['epochs_adaptive'], eta_min=1e-7)

    trainer_jhu = DensityTrainer(
        model=model_jhu, optimizer=opt_jhu, scheduler=sched_jhu,
        loss_fn=CombinedDensityLoss(), device=DEVICE,
        experiment_name='adaptive_csrnet_jhu',
        save_dir=str(CKPT_ROOT), log_dir=str(REPO_ROOT / 'runs'),
    )
    trainer_jhu.load_checkpoint('last.pt')
    history_jhu = trainer_jhu.train(
        jhu_train, jhu_val,
        epochs=CFG['epochs_adaptive'], patience=CFG['patience']
    )

    trainer_jhu.load_checkpoint('best.pt')
    jhu_results = evaluate_density(model_jhu, jhu_test, DEVICE)
    print('\nJHU-Crowd++ Results:')
    for k, v in jhu_results.items():
        print(f'  {k:8s}: {v:.4f}')
else:
    print('JHU-Crowd++ not found at expected path, skipping.')

JHU-Crowd++ dataset found. Training AdaptiveCSRNet on JHU...


JHU Train: 2272 | Val: 500 | Test: 1600
No checkpoint found at /home/ubuntu/crowdvision/checkpoints/adaptive_csrnet_jhu/last.pt

Epoch 1/150
  [TRAIN] loss=0.0080  mae=253.8527
  [VAL  ] mae=120.9697  mse=370.3041  psnr=26.9534  ssim=0.6100  game0=120.9697  game1=34.8515  game2=9.5919  game3=2.6263
  ✓ New best mae: 120.9697

Epoch 2/150
  [TRAIN] loss=0.0070  mae=229.4329
  [VAL  ] mae=136.4286  mse=434.1009  psnr=27.1233  ssim=0.6020  game0=136.4286  game1=36.6755  game2=9.7659  game3=2.6132

Epoch 3/150
  [TRAIN] loss=0.0064  mae=201.1834
  [VAL  ] mae=111.8477  mse=363.4718  psnr=27.3817  ssim=0.6579  game0=111.8477  game1=31.5387  game2=8.5483  game3=2.3265
  ✓ New best mae: 111.8477

Epoch 4/150
  [TRAIN] loss=0.0061  mae=181.2009
  [VAL  ] mae=130.2976  mse=304.7694  psnr=26.8010  ssim=0.6038  game0=130.2976  game1=36.4943  game2=9.7832  game3=2.6523

Epoch 5/150
  [TRAIN] loss=0.0060  mae=179.4024
  [VAL  ] mae=187.0350  mse=445.1778  psnr=27.1858  ssim=0.7596  game0=187.0350  

---
## Density Estimation Complete!

Results and visualisations saved to `experiments/`. Continue with **02_forecasting.ipynb**.